# 第10段階：情報拡散指標の再分析

情報拡散指標を介入後の反応指標として分析する。第2段階の広い探索的相関と、無介入・春学期代表条件・修正済み先行研究高説得力条件・第8段階主候補のpaired比較を分けて確認する。

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = ['Hiragino Sans', 'Noto Sans CJK JP', 'sans-serif']
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.unicode_minus'] = False

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 'experiment_protocols').is_dir() else cwd.parent
STAGE_ROOT = REPO_ROOT / 'experiments/summer_2026/stage10_information_diffusion'
requested_analysis_root = os.environ.get('STAGE10_ANALYSIS_ROOT')
requested_id = os.environ.get('STAGE10_EXPERIMENT_ID')
if requested_analysis_root:
    candidates = [Path(requested_analysis_root).expanduser().resolve()]
elif requested_id:
    candidates = [STAGE_ROOT / requested_id / 'information_diffusion_analysis_v01']
else:
    candidates = sorted(STAGE_ROOT.glob('*/information_diffusion_analysis_v01'), reverse=True)
completed = []
for candidate in candidates:
    manifest_path = candidate / 'analysis_manifest.json'
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest.get('status') == 'completed':
            completed.append(candidate)
if not completed:
    raise FileNotFoundError('Stage 10の完了済み正式分析が見つかりません')
ANALYSIS_ROOT = completed[0]
TABLE_ROOT = ANALYSIS_ROOT / 'tables'
FIGURE_ROOT = ANALYSIS_ROOT / 'figures'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

def read_csv(name):
    return pd.read_csv(TABLE_ROOT / f'{name}.csv')

summary = json.loads((ANALYSIS_ROOT / 'analysis_summary.json').read_text(encoding='utf-8'))
decision = json.loads((ANALYSIS_ROOT / 'decision.json').read_text(encoding='utf-8'))
audit = read_csv('data_audit')
design_corr = read_csv('design_information_correlations')
outcome_corr = read_csv('outcome_information_correlations')
partial_corr = read_csv('partial_correlations')
paired = read_csv('primary_paired_effects')
ratio_diagnostics = read_csv('ratio_diagnostics')
behavior_ts = pd.read_parquet(TABLE_ROOT / 'behavior_time_series.parquet')
information_ts = pd.read_parquet(TABLE_ROOT / 'information_time_series.parquet')

NETWORK_ORDER = ['ba1000', 'facebook', 'wiki_vote']
NETWORK_LABELS = {'ba1000': 'BA1000', 'facebook': 'Facebook', 'wiki_vote': 'Wiki-vote'}
CONDITION_ORDER = ['none', 'legacy_balance', 'prior_high', 'final_candidate']
CONDITION_LABELS = {
    'none': '無介入',
    'legacy_balance': '春学期代表',
    'prior_high': '先行研究高説得力',
    'final_candidate': '第8段階主候補',
}
CONDITION_COLORS = {
    'none': '#4D4D4D',
    'legacy_balance': '#D55E00',
    'prior_high': '#0072B2',
    'final_candidate': '#009E73',
}
PRIMARY_INDICATORS = [
    'misinformation_fst_viewed_per_agent_iteration',
    'corrective_fst_viewed_per_agent_iteration',
    'observational_fst_viewed_per_agent_iteration',
    'behavior_guiding_fst_viewed_per_agent_iteration',
    'corrective_to_misinformation_fst_viewed_ratio',
]
INDICATOR_LABELS = {
    PRIMARY_INDICATORS[0]: '誤情報・初回アクセス',
    PRIMARY_INDICATORS[1]: '訂正情報・初回アクセス',
    PRIMARY_INDICATORS[2]: '観察情報・初回アクセス',
    PRIMARY_INDICATORS[3]: '行動誘導情報・初回アクセス',
    PRIMARY_INDICATORS[4]: '訂正／誤情報・初回アクセス比',
}
INFO_ORDER = ['misinformation', 'corrective', 'observational', 'behavior_guiding']
INFO_LABELS = {
    'misinformation': '誤情報',
    'corrective': '訂正情報',
    'observational': '観察情報',
    'behavior_guiding': '行動誘導情報',
}
print('analysis root =', ANALYSIS_ROOT)
summary

## 1. データ品質と分析範囲

第2段階は探索的相関、固定条件はpaired比較として解釈する。Facebookの主候補は探索的fallbackである。

In [ ]:
display(audit)
display(pd.DataFrame(decision['networks']))
display(ratio_diagnostics[['source', 'network', 'condition_group', 'metric', 'zero_denominator_count', 'numerator_sum', 'denominator_sum', 'ratio_of_sums']].dropna(how='all', axis=1).head(30))

## 2. 設計変数と主情報指標

第2段階のランダムサーチを主に用い、確実性・有効性と初回アクセス指標のSpearman相関を確認する。

In [ ]:
plot_data = design_corr[(design_corr['source'] == 'stage2_saved_trials') & (design_corr['subset'] == 'random_search_primary') & design_corr['indicator'].isin(PRIMARY_INDICATORS)].copy()
predictors = [('applied_certainty', '確実性'), ('applied_effectiveness', '有効性')]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
y = np.arange(len(PRIMARY_INDICATORS))
for row_index, (predictor, predictor_label) in enumerate(predictors):
    for col_index, network in enumerate(NETWORK_ORDER):
        ax = axes[row_index, col_index]
        subset = plot_data[(plot_data['predictor'] == predictor) & (plot_data['network'] == network)].set_index('indicator').reindex(PRIMARY_INDICATORS)
        ax.scatter(subset['rho'], y, color='#0072B2', s=45)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-1.05, 1.05)
        ax.set_title(f'{NETWORK_LABELS[network]} / {predictor_label}')
        ax.set_yticks(y, [INDICATOR_LABELS[value] for value in PRIMARY_INDICATORS])
        ax.set_xlabel(r'Spearman順位相関 $\rho$')
fig.suptitle('設計変数と主情報指標（第2段階ランダムサーチ）', fontsize=15)
fig.tight_layout()
path = FIGURE_ROOT / '01_設計変数と主情報指標.png'
fig.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
path

## 3. 主情報指標と行動結果

単純相関と、確実性・有効性を統制した偏相関を並べる。偏相関が弱まる場合は、情報指標と設計変数の連動を考慮する。

In [ ]:
simple = outcome_corr[(outcome_corr['source'] == 'stage2_saved_trials') & (outcome_corr['subset'] == 'random_search_primary') & outcome_corr['indicator'].isin(PRIMARY_INDICATORS)]
partial = partial_corr[(partial_corr['source'] == 'stage2_saved_trials') & (partial_corr['subset'] == 'random_search_primary') & partial_corr['indicator'].isin(PRIMARY_INDICATORS)]
outcomes = [('j_cum', r'$J_{\mathrm{cum}}$'), ('j_peak', r'$J_{\mathrm{peak}}$')]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
y = np.arange(len(PRIMARY_INDICATORS))
for row_index, (outcome, outcome_label) in enumerate(outcomes):
    for col_index, network in enumerate(NETWORK_ORDER):
        ax = axes[row_index, col_index]
        s = simple[(simple['outcome'] == outcome) & (simple['network'] == network)].set_index('indicator').reindex(PRIMARY_INDICATORS)
        p = partial[(partial['outcome'] == outcome) & (partial['network'] == network)].set_index('indicator').reindex(PRIMARY_INDICATORS)
        ax.scatter(s['rho'], y - 0.12, label='単純相関', color='#0072B2', s=38)
        ax.scatter(p['rho'], y + 0.12, label='偏相関', color='#D55E00', marker='s', s=34)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-1.05, 1.05)
        ax.set_title(f'{NETWORK_LABELS[network]} / {outcome_label}')
        ax.set_yticks(y, [INDICATOR_LABELS[value] for value in PRIMARY_INDICATORS])
        ax.set_xlabel(r'Spearman相関 $\rho$')
axes[0, 2].legend(loc='lower right')
fig.suptitle('主情報指標と行動結果（第2段階ランダムサーチ）', fontsize=15)
fig.tight_layout()
path = FIGURE_ROOT / '02_情報指標と行動結果.png'
fig.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
path

## 4. 最終候補と無介入の主情報反応

差は「第8段階主候補 - 無介入」である。比率と正規化数は単位が異なるため、指標ごとに別軸で表示する。

In [ ]:
effects = paired[(paired['comparison_id'] == 'final_candidate_vs_none') & paired['metric'].isin(PRIMARY_INDICATORS)].copy()
fig, axes = plt.subplots(1, 5, figsize=(21, 4.8), sharey=True)
y = np.arange(len(NETWORK_ORDER))
for ax, metric in zip(axes, PRIMARY_INDICATORS):
    subset = effects[effects['metric'] == metric].set_index('network').reindex(NETWORK_ORDER)
    estimate = subset['candidate_minus_reference'].to_numpy(float)
    low = subset['ci_low'].to_numpy(float)
    high = subset['ci_high'].to_numpy(float)
    ax.errorbar(estimate, y, xerr=np.vstack([estimate - low, high - estimate]), fmt='o', color='#009E73', capsize=4)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(INDICATOR_LABELS[metric], fontsize=10)
    ax.set_yticks(y, [NETWORK_LABELS[value] for value in NETWORK_ORDER])
    ax.set_xlabel('主候補 - 無介入')
fig.suptitle('第8段階主候補の情報反応差と95% CI', fontsize=15)
fig.tight_layout()
path = FIGURE_ROOT / '03_主候補と無介入の情報反応差.png'
fig.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
display(effects[['network', 'metric', 'candidate_minus_reference', 'ci_low', 'ci_high', 'interpretation']])
path

## 5. 累積利己的行動割合の時系列

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=False)
for ax, network in zip(axes, NETWORK_ORDER):
    for condition in CONDITION_ORDER:
        subset = behavior_ts[(behavior_ts['network'] == network) & (behavior_ts['condition_group'] == condition)].sort_values('t')
        x = subset['t'].to_numpy(float)
        mean = subset['mean'].to_numpy(float)
        low = subset['ci_low'].to_numpy(float)
        high = subset['ci_high'].to_numpy(float)
        ax.plot(x, mean, label=CONDITION_LABELS[condition], color=CONDITION_COLORS[condition])
        ax.fill_between(x, low, high, color=CONDITION_COLORS[condition], alpha=0.13)
    ax.set_title(NETWORK_LABELS[network])
    ax.set_xlabel('時刻')
    ax.set_ylabel('累積利己的行動割合')
axes[-1].legend(loc='best', fontsize=9)
fig.suptitle('4条件の累積利己的行動割合（平均と95% CI）', fontsize=15)
fig.tight_layout()
path = FIGURE_ROOT / '04_累積利己的行動割合の時系列.png'
fig.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
path

## 6. 情報種別の累積初回アクセス数

縦軸は1エージェント当たりの累積初回アクセス数であり、ユニーク閲覧エージェント数ではない。

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14), sharex='col', sharey=False)
for row_index, info_name in enumerate(INFO_ORDER):
    for col_index, network in enumerate(NETWORK_ORDER):
        ax = axes[row_index, col_index]
        for condition in CONDITION_ORDER:
            subset = information_ts[(information_ts['network'] == network) & (information_ts['condition_group'] == condition) & (information_ts['info_name'] == info_name)].sort_values('t')
            x = subset['t'].to_numpy(float)
            mean = subset['mean'].to_numpy(float)
            low = subset['ci_low'].to_numpy(float)
            high = subset['ci_high'].to_numpy(float)
            ax.plot(x, mean, label=CONDITION_LABELS[condition], color=CONDITION_COLORS[condition], linewidth=1.5)
            ax.fill_between(x, low, high, color=CONDITION_COLORS[condition], alpha=0.10)
        ax.set_title(f'{NETWORK_LABELS[network]} / {INFO_LABELS[info_name]}', fontsize=10)
        ax.set_xlabel('時刻')
        ax.set_ylabel('1エージェント当たり累積初回アクセス')
axes[0, -1].legend(loc='best', fontsize=8)
fig.suptitle('情報種別の累積初回アクセス数（平均と95% CI）', fontsize=15)
fig.tight_layout()
path = FIGURE_ROOT / '05_情報種別の累積初回アクセス数.png'
fig.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
path

## 7. 解釈上の制限

- 情報指標は介入後の反応であり、因果的な媒介効果を直接推定していない。
- 訂正／誤情報比率を設計変数や最適化目的に格上げしない。
- 行動誘導情報の拡散が少ないこと自体を望ましいとはみなさない。
- Facebookの主候補は第7段階の探索的fallbackであり、結果も探索的に扱う。
- 第10段階の結果で候補の再選択は行わない。